# Kepler Orbits: Math for astragent

Working notebook for the math behind astragent's simulator. Background: Calc 1/2 and discrete math about ten years ago, little trig or linear algebra since. Section 0 rebuilds the tools; later sections use them to derive the orbital mechanics `src/` needs.

Each domain section (1+) ends by checking its result against a real reference value — usually JPL Horizons, via `docs/research/2026-09-18-kepler-validation.py`. Section 0's checks are small hand-worked exercises instead, in the same `TODO(human)` style as the rest of the project.

Scope: Phase 1 only (orbit propagation). No Three.js/camera/quaternion math yet — that's a later phase.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 0. Foundations

Rebuilding four tools Sections 2-5 lean on: Newton-Raphson (new algorithm), unit-circle trig, vectors/matrices as transforms, and rotation matrices (mostly refreshers). Order: Newton-Raphson first since it's self-contained, then the trig/rotation pieces build on each other.

### 0.1 Newton-Raphson

The root-finding method behind `kepler.ts`. First, what `kepler.ts` is actually solving:

A planet orbits the sun on an ellipse. "Where is it right now" has three different angle descriptions, each with its own name:

- **Mean anomaly ($M$)** — a fictional angle that increases at a constant rate over time, as if the planet moved at constant speed around a circle. Not physically real, but trivial to compute from time — it's the simulator's input.
- **Eccentric anomaly ($E$)** — an angle on an auxiliary circle that circumscribes the real ellipse. Also not the true physical angle, but once known, the planet's actual position in the orbital plane drops out from simple formulas (Section 4).
- **True anomaly** — the real angle from the sun to the planet. Not needed directly in Phase 1A; position falls out from $E$ first.

So: you know $M$ (from time), you want $E$ (to get position), and the relationship between them is Kepler's equation:

$$M = E - e\sin E$$

where $e$ is the orbit's eccentricity (0 = circle, closer to 1 = more elongated). This can't be solved for $E$ algebraically — $E$ appears both alone and inside a sine — so the simulator finds it iteratively, with Newton-Raphson.

The idea: given a function $f$ and a starting guess $x_0$, use the tangent line at $x_0$ to get a better guess. The tangent line at $x_0$ is $y = f(x_0) + f'(x_0)(x - x_0)$. Setting $y = 0$ and solving for $x$ gives the next guess:

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}
$$

Repeat until $f(x_n)$ is close enough to zero.



In [11]:
import math

def newton_raphson(f, f_prime, x0, tolerance=1e-12, max_iterations=50):
    return x0 - (f / f_prime)

def foo(x):
    return (x*x) - 2

def foo_prime(x):
    return 2 * x

x = 1
for i in range(5):
    f = foo(x)
    f_prime = foo_prime(x)
    res = newton_raphson(f, f_prime, x)
    x = res

print("sqrt of x: ", x)

# now Kepler's equation

def kepler(E, M, e):
    return M - (E - (e * math.sin(E)))

def kepler_prime(E, M, e):
    return (e*math.cos(E)) - 1

E = 1
for i in range(5):
    M = 1
    e = 0.1
    f = kepler(E, M, e)
    f_prime = kepler_prime(E, M, e)
    res = newton_raphson(f, f_prime, E)
    E = res

print("Kepler: ", E)
    

sqrt of x:  1.4142135623730951
Kepler:  1.0885977523978936


### TODO(human)

Work an example by hand first, then in code: find $\sqrt{2}$ using Newton-Raphson on $f(x) = x^2 - 2$ (so $f'(x) = 2x$), starting from $x_0 = 1$.

1. By hand: compute $x_1$ and $x_2$ using the formula above. How many digits of $\sqrt{2} \approx 1.41421356$ do you have after two steps?
2. Implement `newton_raphson` above and confirm it converges to the same value.
3. Apply it directly to Kepler's equation: for $M = 1$ radian, $e = 0.1$, find $E$ such that $E - e\sin E = M$. Here $f(E) = M - (E - e\sin E)$, the residual you're driving to zero. What is $f'(E)$?

In [ ]:
# TODO(human): hand-worked x1, x2 for sqrt(2), then newton_raphson(...) to confirm,
# then solve Kepler's equation for M=1, e=0.1

### 0.2 Unit-circle trig, vectors/matrices as transforms, rotation matrices

_Not written yet. Needed for `position.ts`: the orbital-plane parameterization (unit circle) and the ecliptic rotation (composed rotation matrices)._

## 1. Julian dates & time

A Julian date (JD) is a continuous day count — with a fractional part for time of day — since noon UTC on 1 January 4713 BC. Astronomers use it instead of calendar dates because orbital equations need to consume elapsed time as a single subtraction, not a walk through months, years, and leap days.

It starts at **noon**, not midnight, so a single night of observation never crosses a date boundary.

The reference epoch **J2000.0** is $JD = 2451545.0$, noon UTC on 1 January 2000. Orbital elements (Section 2) are usually given relative to an epoch expressed this way, and `src/`'s simulator will take elapsed days from an epoch as its time input — see `docs/definitions.md`.

### Converting a Gregorian calendar date to JD

One standard algorithm (valid for the Gregorian calendar, i.e. any date we care about here):

$$
\begin{aligned}
a &= \left\lfloor \frac{14 - \text{month}}{12} \right\rfloor \\
y &= \text{year} + 4800 - a \\
m &= \text{month} + 12a - 3 \\
JDN &= \text{day} + \left\lfloor \frac{153m + 2}{5} \right\rfloor + 365y + \left\lfloor \frac{y}{4} \right\rfloor - \left\lfloor \frac{y}{100} \right\rfloor + \left\lfloor \frac{y}{400} \right\rfloor - 32045
\end{aligned}
$$

$JDN$ is the Julian Day *Number* — an integer, valid at noon UTC that day. To get a fractional JD for a specific UTC time, add the fraction of a day elapsed since the *previous* midnight, offset by half a day (because JD ticks over at noon, not midnight):

$$
JD = JDN + \frac{\text{hour} - 12}{24} + \frac{\text{minute}}{1440} + \frac{\text{second}}{86400}
$$

In [2]:
def julian_date(year, month, day, hour=0, minute=0, second=0):
    a = (14 - month) // 12
    y = year + 4800 - a
    m = month + 12 * a - 3
    jdn = day + (153 * m + 2) // 5 + 365 * y + y // 4 - y // 100 + y // 400 - 32045
    return jdn + (hour - 12) / 24 + minute / 1440 + second / 86400

# J2000.0 epoch: noon UTC, 1 January 2000 -> should be exactly 2451545.0
julian_date(2000, 1, 1, 12, 0, 0)

2451545.0

### Check it against a second known value

`docs/definitions.md` already states: midnight UTC on 18 September 2026 is $JD = 2461301.5$. Confirm `julian_date` agrees.

In [3]:
computed = julian_date(2026, 9, 18, 0, 0, 0)
expected = 2461301.5

print(f"computed : {computed}")
print(f"expected : {expected}")
assert computed == expected

computed : 2461301.5
expected : 2461301.5


### TODO(human)

How many days elapsed between the J2000.0 epoch and midnight UTC on your own birthday this year (or any date of your choosing)? Compute it using `julian_date` — two JDs subtract directly into elapsed days, which is the entire reason astronomers use this system.

In [ ]:
# TODO(human): compute elapsed days between J2000.0 (2451545.0)
# and midnight UTC on a date of your choosing, using julian_date().
elapsed_days = ...

elapsed_days

## 2. Orbital elements

_Not written yet._

## 3. Kepler's equation (mean anomaly → eccentric anomaly)

_Not written yet._

## 4. Eccentric anomaly → perifocal position

_Not written yet._

## 5. Perifocal → ecliptic rotation

_Not written yet._

## 6. End-to-end validation

_Not written yet — will reuse the Icarus/Horizons comparison from `docs/research/2026-09-18-kepler-validation.py`._